# Executive Email Generation Project

## Objective
Clean a raw dataset of executives, identify senior leadership, and generate two likely corporate email addresses for 50 executives.

Steps:
- Load dirty CSV
- Filter senior executives
- Remove non-executive roles
- Strip duplicate records
- Rank by seniority
- Select top 50
- Add domains
- Generate email formats
- Export final list


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display


## 2. Load Raw Dataset

In [2]:
df=pd.read_csv("cmo_videos_names.csv",on_bad_lines="skip")
df.head()

,Name,Title,Company,Youtube URL
0,Donna Johnson,Chief Marketing Officer,Inseego,https://www.youtube.com/watch?v=mwlCTdYh0uM
1,Andreas Urschitz,CMO,Infineon,https://www.youtube.com/watch?v=QUgwtmPtN8A
2,Robert Occhialini,Chief Technology Officer,World Surf League,https://www.youtube.com/watch?v=VkMVaML9yBc
3,Gabriel Romero,Chief Marketing Officer,AllCloud,https://www.youtube.com/watch?v=VkMVaML9yBc
4,Julie Neumann,Chief Marketing Officer,Honeycomb.io,https://www.youtube.com/watch?v=mIMHw1U2BPg


## 3. Filter Senior Executives and remove non execcutive roles

In [3]:
senior_titles=["CEO","Founder","Co-Founder","President",
    "COO","CFO","CTO","CMO","CIO",
    "Vice President","VP","SVP","EVP",
    ]
non_exec_titles=["Engineer","Developer","Analyst","Associate","Intern","Student","Assistant","Coordinator","Specialist","Consultant","Sales Rep",
               "Sales Representative","Marketing Executive","HR Executive","Recruiter","Talent","Support","Administrator","Admin",
               "Executive Assistant","Operations Associate","Director","Manager","Sr.","Senior Manager","GM","General Manager",
    "Managing Director",]
execs=df[df["Title"].str.contains("|".join(senior_titles), case=False, na=False)]
execs = execs[~execs["Title"].str.contains("not stated|unknown|n/a", case=False, na=False)]
execs = execs[~execs["Title"].str.contains("|".join(non_exec_titles), case=False)]
execs[["Name","Title"]]


,Name,Title
1,Andreas Urschitz,CMO
14,Michael Nicosia,Co-Founder & COO
19,Mick Hollison,Founder & CEO
20,Bernd Greifeneder,Chief Technology Officer & Founder
28,Danielle Goode Coady,VP of Marketing
...,...,...
337,Randy Arseneau,CMO
341,Marie Hattar,CMO
344,Steve Pao,CMO
352,Randy Arsenau,CMO


## 4. Remove Duplicate Records

In [15]:
execs = execs.drop_duplicates(subset=["Name","Company"]) 

## 5. Remove Evangelist / Field Roles

In [16]:
execs = execs[~execs["Title"].str.contains("evangelist", case=False, na=False)] #because sometimes they are not budget owners
execs = execs[~execs["Title"].str.contains("field", case=False, na=False)]      #they usually report to CTO who are the actual executive

## 6. Rank Executives by Seniority

In [17]:
def rank(title):
    t = title.lower()

    if "ceo" in t or "founder" in t or "president" in t:
        return 3

    if any(x in t for x in ["cmo","cto","cfo","coo","cio","chief marketing","chief technology","chief financial","chief operating","chief information"]):
        return 2

    if any(x in t for x in ["evp","svp","vp","gvp"]):
        return 1

    return 0
execs["rank"] = execs["Title"].apply(rank)
execs = execs.sort_values(by=["rank", "Name"], ascending=[False, True])
execs.head(20)

,Name,Title,Company,Youtube URL,rank
51,Abhay Parasnis,Founder & CEO,Typeface,https://www.youtube.com/watch?v=U1jdmFW7-f4,3
333,Ben Parr,Co-Founder & CMO,Octane AI,https://www.youtube.com/watch?v=riKER3eQixs,3
20,Bernd Greifeneder,Chief Technology Officer & Founder,Dynatrace,https://www.youtube.com/watch?v=6bBI-WZYvTo,3
132,Bill Largent,CEO,Veeam,https://www.youtube.com/watch?v=OdrhPIBgZoE,3
215,Boris Renski,Co Founder and CMO,Mirantis,https://www.youtube.com/watch?v=EcxoQizrOPw,3
152,Brandon Nott,Senior Vice President of Product,UiPath,https://www.youtube.com/watch?v=SJFpjG_J6W0,3
83,Brett Hannath,Corporate Vice President & CMO,Intel,https://www.youtube.com/watch?v=sAPATcXqel0,3
178,Calline Sanchez,Vice President,IBM Worldwide Systems Lab Services,https://www.youtube.com/watch?v=DEHUqmEwFeg,3
131,Carol Carpenter,Senior Vice President and Chief Marketing Officer,VMware,https://www.youtube.com/watch?v=5KcRXgaimYU,3
314,Jaspreet Singh,CEO,Druva,https://www.youtube.com/watch?v=cf1vPcFW5to,3


In [18]:
top50 = execs.head(50)
top50 = top50.reset_index(drop=True)
top50.index=top50.index+1
top50["Name"] = top50["Name"].str.replace(r'[^\w\s]', '', regex=True)
top50

,Name,Title,Company,Youtube URL,rank
1,Abhay Parasnis,Founder & CEO,Typeface,https://www.youtube.com/watch?v=U1jdmFW7-f4,3
2,Ben Parr,Co-Founder & CMO,Octane AI,https://www.youtube.com/watch?v=riKER3eQixs,3
3,Bernd Greifeneder,Chief Technology Officer & Founder,Dynatrace,https://www.youtube.com/watch?v=6bBI-WZYvTo,3
4,Bill Largent,CEO,Veeam,https://www.youtube.com/watch?v=OdrhPIBgZoE,3
5,Boris Renski,Co Founder and CMO,Mirantis,https://www.youtube.com/watch?v=EcxoQizrOPw,3
6,Brandon Nott,Senior Vice President of Product,UiPath,https://www.youtube.com/watch?v=SJFpjG_J6W0,3
7,Brett Hannath,Corporate Vice President & CMO,Intel,https://www.youtube.com/watch?v=sAPATcXqel0,3
8,Calline Sanchez,Vice President,IBM Worldwide Systems Lab Services,https://www.youtube.com/watch?v=DEHUqmEwFeg,3
9,Carol Carpenter,Senior Vice President and Chief Marketing Officer,VMware,https://www.youtube.com/watch?v=5KcRXgaimYU,3
10,Jaspreet Singh,CEO,Druva,https://www.youtube.com/watch?v=cf1vPcFW5to,3


## 8. Add Company Domains (Manual Verification)

Company domains were manually verified by visiting each company’s official website and entering the correct email domain.  
This ensures higher accuracy than automated guessing.


In [23]:
# Create empty domain column for manual filling
top50["domain"] = ""

# Export for manual domain enrichment
top50.to_csv("top50_with_domains.csv", index=False)


In [26]:
# Reload CSV after domains have been manually added
top50 = pd.read_excel("top50_with_domains.xlsx")
top50.head()

,Name,Title,Company,Youtube URL,rank,first,last,domain
0,Abhay Parasnis,Founder & CEO,Typeface,https://www.youtube.com/watch?v=U1jdmFW7-f4,3,abhay,parasnis,typeface.ai
1,Ben Parr,Co-Founder & CMO,Octane AI,https://www.youtube.com/watch?v=riKER3eQixs,3,ben,parr,octaneai.com
2,Bernd Greifeneder,Chief Technology Officer & Founder,Dynatrace,https://www.youtube.com/watch?v=6bBI-WZYvTo,3,bernd,greifeneder,dynatrace.com
3,Bill Largent,CEO,Veeam,https://www.youtube.com/watch?v=OdrhPIBgZoE,3,bill,largent,veeam.com
4,Boris Renski,Co Founder and CMO,Mirantis,https://www.youtube.com/watch?v=EcxoQizrOPw,3,boris,renski,mirantis.com


## 8. Generate Email Addresses


In [27]:
top50["first"] = top50["Name"].str.split().str[0].str.lower()
top50["last"] = top50["Name"].str.split().str[-1].str.lower()


In [28]:
top50["email1"] = top50["first"] + "." + top50["last"] + "@" + top50["domain"]
top50["email2"] = top50["first"].str[0] + top50["last"] + "@" + top50["domain"]
top50 = top50.drop(columns=["first","last","rank"])
top50.index=top50.index+1
top50["Company"] = top50["Company"].str.title()
top50

,Name,Title,Company,Youtube URL,domain,email1,email2
1,Abhay Parasnis,Founder & CEO,Typeface,https://www.youtube.com/watch?v=U1jdmFW7-f4,typeface.ai,abhay.parasnis@typeface.ai,aparasnis@typeface.ai
2,Ben Parr,Co-Founder & CMO,Octane Ai,https://www.youtube.com/watch?v=riKER3eQixs,octaneai.com,ben.parr@octaneai.com,bparr@octaneai.com
3,Bernd Greifeneder,Chief Technology Officer & Founder,Dynatrace,https://www.youtube.com/watch?v=6bBI-WZYvTo,dynatrace.com,bernd.greifeneder@dynatrace.com,bgreifeneder@dynatrace.com
4,Bill Largent,CEO,Veeam,https://www.youtube.com/watch?v=OdrhPIBgZoE,veeam.com,bill.largent@veeam.com,blargent@veeam.com
5,Boris Renski,Co Founder and CMO,Mirantis,https://www.youtube.com/watch?v=EcxoQizrOPw,mirantis.com,boris.renski@mirantis.com,brenski@mirantis.com
6,Brandon Nott,Senior Vice President of Product,Uipath,https://www.youtube.com/watch?v=SJFpjG_J6W0,uipath.com,brandon.nott@uipath.com,bnott@uipath.com
7,Brett Hannath,Corporate Vice President & CMO,Intel,https://www.youtube.com/watch?v=sAPATcXqel0,intel.com,brett.hannath@intel.com,bhannath@intel.com
8,Calline Sanchez,Vice President,Ibm Worldwide Systems Lab Services,https://www.youtube.com/watch?v=DEHUqmEwFeg,ibm.com,calline.sanchez@ibm.com,csanchez@ibm.com
9,Carol Carpenter,Senior Vice President and Chief Marketing Officer,Vmware,https://www.youtube.com/watch?v=5KcRXgaimYU,vmware.com,carol.carpenter@vmware.com,ccarpenter@vmware.com
10,Jaspreet Singh,CEO,Druva,https://www.youtube.com/watch?v=cf1vPcFW5to,druva.com,jaspreet.singh@druva.com,jsingh@druva.com


In [ ]:
top50.to_csv("final_exec_emails.csv", index=False)
